# **Project title:** Quantitative assessment of slc35a2 localization to the Golgi in single cells
### **Researcher:** Xavier Williams
### **Notebook author:** Shannon Rhoads (srhoads@unc.edu)

### **Editing & version log:**
- v0.1: 20260115 - downloaded [notebook 1.1_infer_masks_from-composite_with_nuc.ipynb](https://github.com/SCohenLab/infer-subc/blob/v2.0.0b1/notebooks/part_1_segmentation_workflows/1.1_infer_masks_from-composite_with_nuc.ipynb) from [infer-subc version v2.0.0b1](https://github.com/SCohenLab/infer-subc/tree/v2.0.0b1)
- v0.2: 20260115 - the following edits were made:
    - created new conda environment for this project with the following:
        - conda create -n golgi-coloc python=3.10
        - pip install infer-subc
        - pip install bioio bioio-czi
        - pip install napari[all]
    - Update image ready to BioImageIO package
    - Nuclei declumping methods were added to better separate neighboring nuclei
    - Filtering step to remove nuclei that are extremely bright (these are usually rounded and likely dividing or dying)

----------
## **How to use Jupyter notebooks**

Advance through each block of code below by pressing `Shift`+`Enter` or pressing the "Execute Cell" (`▶️`) button to the left of each block.

You will see a series of instructions before each block of code. Be on the look out for the following headers and follow the instructions accordingly:
- &#x1F3C3; **Run code; no user input required** - proceed without adding anything to the code block
- &#x1F453; **FYI** (for your information) - helpful information usually to bring context to what is going on
- &#x1F6D1; &#x270D; **User Input Required** - stop and input the appropriate information about your data. The following indicator will also be present in the code block:
   ```python 
   #### USER INPUT REQUIRED ###
   ```

----------

# **Notebook title:** Infer golgi

### **Purpose:**
To determine the most appropriate settings to segment the Golgi from fluorescence microscopy images. These settings will be applied to a batch of images in the batch_process_segmentation notebook.

### 👣 **Summary of analysis steps**  

➡️ **INPUT**

In this workflow, a single or multi-channel confocal microscopy image of fluorescently tagged organelles will be used to "infer" (or segment) the Golgi.

➡️ **EXTRACTION**
- **`STEP 1`** - Select a channel for segmentation

    - select single channel containing the golgi marker (channel number = user input)

**PRE-PROCESSING**
- **`STEP 2`** - Rescale and smooth image

  - rescale intensity of composite image (min=0, max=1)
  - median filter (median size = user input)
  - gaussian filter (sigma = user input)

**CORE PROCESSING**
- **`STEP 3`** - Global + local thresholding (AICSSeg – MO)

    - apply MO thresholding method from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (threshold options = user input)

- **`STEP 4`** - Thin segmentation (Topology preserving)

    - apply topology preserved thinning function from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (thinning amount, minimum thickness = user input)

- **`STEP 5`** - ‘Dot’ thresholding method (AICSSeg)

    - apply "dot" thresholding method (for small round objects) from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (size scale and threshold cutoff = user input)

- **`STEP 6`** - Combine Segmentations (logical or)

    - combine the two segmentations with logical *OR*

**POST-PROCESSING**
- **`STEP 7`** - Remove small holes and objects

  - fill holes (hole size = user input)
  - remove small objects (object size = user input)
  - filter method (method = user input)

**POST-POST-PROCESSING**
- **`STEP 8`** - Label objects

  - label unique golgi objects based on connectivity or declumping

**EXPORT** ➡️
- save labeled ***golgi*** (golgi, GL) as unsigned integer 16-bit tif files


*This workflow is an adaptation of the Allen Cell Segmenter procedure for segmentation of Golgi from the [sialyltransferase 1 (ST6GAL1)](https://www.allencell.org/cell-observations/category/golgi-apparatus) marker. Sourced from: this [script](https://github.com/AllenCell/aics-segmentation/blob/main/aicssegmentation/structure_wrapper/seg_st6gal1.py).*

---------------------
## **IMPORTS AND LOAD IMAGE**

Details about the functions included in this subsection are outlined in the [`1.0_image_setup`](1.0_image_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

#### &#x1F3C3; **Run code; no user input required**

In [32]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import napari
from napari.utils.notebook_display import nbscreenshot

from aicssegmentation.core.utils import topology_preserving_thinning

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files,
                                     sample_input)
from infer_subc.core.img import *

from bioio import BioImage

import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.segmentation import (highpass_filter, otsu_size_filter, watershed_declumping, infer_golgi)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `im_type`: the file type of your image written in quotation marks; this must be written within quotation markers. 
    - EX: ".czi" or ".tiff"
- `data_root_path`: the path to folder that your input data and a separate folder for segmentation outputs; this must be written within quotation markers. If you are using a Windows computer, paths are copied with backslashes ("\"). Python will be confused by this; switch these to forward slashes ("/") or place an "r" in front of the path string.  
    - EX: "C:/Users/{your-user-name}/Documents/Exp1" or r"C:\Users\{your-user-name}\Documents\Exp1"
- `raw_data_path`: the path to the folder that contains your input data; this must be written within quotation markers. 
    - EX: data_root_path / "input"
- `seg_data_path`: the path to the folder where segmentation output files will be saved; this must be written within quotation markers. EX: data_root_path / "segmentations". If this folder does not exist already, it will be created for you in the subsequent lines of code.

In [2]:
#### USER INPUT REQUIRED ###
im_type = ".czi"
data_root_path = Path(os.path.dirname(os.getcwd())) 
raw_data_path = Path(data_root_path) / "pilot-data"
seg_data_path = Path(data_root_path) / "test-segmentation"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** A list of the images included in the `raw_data_path` folder is printed below for easy reference below.

In [5]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(seg_data_path):
    Path.mkdir(seg_data_path)
    print(f"Segmentation data path did not exist; making it now: {seg_data_path}")

# list files in the input folder
print(f"Listing all {im_type} files in the raw data path: ")
img_file_list = list_image_files(Path(raw_data_path),im_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":img_file_list})

Listing all .czi files in the raw data path: 


,Image Name
0,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026A88PB1T1P1-01.czi
1,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026GLN108STOPB1T1P1-08.czi
2,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026R55LB1T1P1-01.czi


#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: 
- `test_img_n`: the index, or number, associated with your image of choice. This value can be found to the left of the image path listed in the table above.
- `channel_names`: specify the name of each channel in your image in order. The name of each channel should be within quotation markers and separated by a common. The entire list should be within square brackets ([ ]) -- a Python list.
    - EX: ['DAPI', 'slc35a2', 'golgi']

In [ ]:
#### USER INPUT REQUIRED ###
test_img_n = 1
channel_names = ['golgi', 'slc35a2', 'DAPI']

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block reads the image and image metadata into memory. The metadata is printed and the image is visualized in Napari. A new window will appear displaying the Napari Viewer.

*IMPORTANT NOTES:*
1. *The code block below is "hard-coded" to process 3D images.* 
2. *Metadata is extracted correctly based on the .czi files. If file type changes, the metadata structure may also change in impact the values below. The most important variables to update from those below are the `dim_size` which is used to derive the scale (voxel size) and the `channel_axis`.*
3. *If you image type is a .czi image series (multiple FOVs in one file), the image import method used below will ONLY READS THE FIRST FOV.*

In [48]:
# read in image and metadata
print(f"Reading in image and metadata for {img_file_list[test_img_n]}...")

file_path = str(img_file_list[test_img_n])
raw_file = BioImage(file_path)
img_data = np.squeeze(raw_file.data) # np.squeeze removes single-dimensional entries from the shape of an array
meta_dict = raw_file.standard_metadata

# extra metadata information
dimensions = meta_dict.dimensions_present
channel_axis = str(dimensions).index('C')
dim_size = {"Z": raw_file.physical_pixel_sizes.Z, 
            "Y": raw_file.physical_pixel_sizes.Y, 
            "X": raw_file.physical_pixel_sizes.X}
# select the scale values from the dim_size dict in the order in which X, Y, and Z appear in the dimensions string
scale_order = [dim for dim in str(dimensions) if dim in ['Z','Y','X']]
scale = tuple([dim_size[dim] for dim in scale_order])

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale ({scale_order}): {scale}")
print(f"Channel axis: {channel_axis}")

# open viewer and add images
viewer = napari.Viewer()
for i, channel_name in enumerate(channel_names):
    viewer.add_image(img_data[i],
                     scale=scale,
                     name=f"Channel {i}: {channel_name}")
viewer.grid.enabled = True
viewer.reset_view()
print("\nProceed to Napari window to view your selected image.")

Reading in image and metadata for c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026A88PB1T1P1-01.czi...


Failed to parse position index from scene name 'Image:0': invalid literal for int() with base 10: 'mage:0'
Traceback (most recent call last):
  File "c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\bioio_czi\standard_metadata.py", line 20, in position_index
    return int(prefix[1:])
ValueError: invalid literal for int() with base 10: 'mage:0'


Metadata information
File path: c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026A88PB1T1P1-01.czi
Channel 0 name: golgi
Channel 1 name: slc35a2
Channel 2 name: DAPI
Scale (['Z', 'Y', 'X']): (0.2, 0.07059082892416223, 0.07059082892416223)
Channel axis: 0



Proceed to Napari window to view your selected image.


-----
## **EXTRACTION**

### **`STEP 1` - Select a channel for segmentation**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify which channel includes your Golgi label:
- `GOLGI_CH`: the index of the channel containing your Golgi label. Image indexing begins with 0, not 1. Reference the channel numbers indicated in the Napari window for easy reference.

In [8]:
#### USER INPUT REQUIRED ###
GOLGI_CH = channel_names.index('golgi')

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block extracts the Golgi channel from your multi- (or single) channel image. It will be the only part of the image used in the rest of this workflow. The single golgi channel is added to the Napari viewer.

In [9]:
# select channel
raw_golgi = select_channel_from_raw(img_data, GOLGI_CH)

# clear napari and add single channel as a new layer
viewer.layers.clear()
viewer.grid.enabled = False
viewer.add_image(raw_golgi, scale=scale, name="1 - Extract Golgi")

<Image layer '1 - Extract Golgi' at 0x19df81cca30>

-----
## **PRE-PROCESSING**

### **`STEP 2` - Rescale and smooth image**

&#x1F453; **FYI:** This code block rescales the image so that the pixel/voxel with the highest intensity is set to 1 and the one with the lowest intensity is set to 0. The image is then *optionally* smoothed using a Gaussian and/or median filter. 


#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the amount of filter to use for each method. Higher values indicate more smoothing:
- `med_filter_size`: the size of the median filter to apply; if 0 is used, no filter will be applied
- `gaussian_smoothing_sigma`: the sigma to apply in the Gaussian filtering step; if 0 is used, no filter will be applied

In [10]:
#### USER INPUT REQUIRED ###
median_sz = 4
gauss_sig = 1.4

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block rescales the image and applies the specified median and Gaussian filters. The image is then added to Napari as a new layer for visual comparison to the input image. 

Use the Napari viewer to iteratively adjust the smoothing settings selected above.

In [11]:
# rescaling and smoothing input image
struct_img =  scale_and_smooth(raw_golgi,
                               median_size = median_sz, 
                               gauss_sigma = gauss_sig)

# adding image to Napari as a new layer
viewer.add_image(struct_img, scale=scale, name=f"2 - Rescale and Smooth (median_sz={median_sz}, gauss_sig={gauss_sig})")

<Image layer '2 - Rescale and Smooth (median_sz=4, gauss_sig=1.4)' at 0x19de5024ca0>

-----
## **CORE-PROCESSING**

### **`STEP 3` - Global + local thresholding (AICSSeg – MO)**


&#x1F453; **FYI:** This code block is the first of two semantic segmentation steps that are combined together in a later step, and is used to segment Golgi cisterae strucutres. `Semantic segmentation` is the process of deciding whether a pixel/voxel should be included in an object (labeled with a value of 1) or should be considered as part of the background (labeled with a value of 0). A semantic segmentation does not discern individual objects from one another.

The masked_object_filter utilizes the 'MO' filter from the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. AICS documentation states: "The algorithm is a hybrid thresholding method combining two levels of thresholds. The steps are: [1] a global threshold is calculated, [2] extract each individual connected componet after applying the global threshold, [3] remove small objects, [4] within each remaining object, a local Otsu threshold is calculated and applied with an optional local threshold adjustment ratio (to make the segmentation more and less conservative). An extra check can be used in step [4], which requires the local Otsu threshold larger than 1/3 of global Otsu threhsold and otherwise this connected component is discarded."

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `thresh_method`: the global thresholding method; options include 'tri'/'triangle, 'med'/'median' or 'ave'/'ave_tri_med'. Triangle implements the [skimage.filters.threshold_triangle](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.threshold_triangle) method, median utilizes the 50th percentile of the intensities, and ave uses the average of the two methods to calculate the lower bound of the global threshold.
- `cell_wise_min_area`: the minimum expected size of your object; smaller objects will be removed prior to local thresholding
- `thresh_adj`: adjustment to make to the local threshold; larger values make the segmentation more stringent (less area included)

In [12]:
#### USER INPUT REQUIRED ###
thresh_method = 'median'
cell_wise_min_area = 1200
thresh_adj = 1.3

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the MO filter using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [13]:
# segment the majority of the golgi with this global and local thresholding method
bw = masked_object_thresh(struct_img, 
                          global_method=thresh_method, 
                          cutoff_size=cell_wise_min_area, 
                          local_adjust=thresh_adj)

# adding image to Napari as a new layer
viewer.add_image(bw, scale=scale, name=f"3 - MO filter (thresh_method={thresh_method}, thresh_adj={thresh_adj}, size={cell_wise_min_area})", opacity=0.3, colormap="cyan", blending='additive')

<Image layer '3 - MO filter (thresh_method=median, thresh_adj=1.3, size=1200)' at 0x19de5027a00>

### **`STEP 4` - Thin segmentation (topology preserving)**

&#x1F453; **FYI:** This code block implements a thinning step to refine the result of the MO segmentation. This logic was developed in the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. The [topology_preserve_thinning](https://github.com/AllenCell/aics-segmentation/blob/main/aicssegmentation/core/utils.py#L101) function reduces the diameter of the segmented object while preserving its overall topology.

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `thin_dist_preserve`: half of the minimum width of your object
- `thin_dist`: the amount to thin by in pixels/voxels; must be a positive integer; a value of 0 can be used to if no thinning is needed

In [14]:
#### USER INPUT REQUIRED ###
thin_dist_preserve = 5
thin_dist = 2

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the thinning step using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [15]:
# thin segmentation with maintain a minimal required thickness
bw_thin = topology_preserving_thinning(bw, thin_dist_preserve, thin_dist)

# adding image to Napari as a new layer
viewer.add_image(bw_thin, scale=scale, name=f"4 - Thinning (preserve={thin_dist_preserve}, thin_dist={thin_dist})", opacity=0.3, colormap="magenta", blending='additive')

<Image layer '4 - Thinning (preserve=5, thin_dist=2)' at 0x19e03c57490>

### **`STEP 5` -  ‘Dot’ thresholding method (AICSSeg)**

&#x1F453; **FYI:** This code block is the second of two semantic segmentation steps that are combined together in a later step, and is used to segment Golgi-derived vesicles (circular/spherical objects). The 'dot' filter is derived from the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. It utilizes up to three scale (object size) and cutoff (threshold) pairs for objects of different size and intensity. This function is specifically designed to segment small round objects.

This step may not be necessary if the previous step was able to segment the entire Golgi well enough. If you do not want to apply any segmenation method using this approach, set all values below to 0.

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the scale and cutoff values for each pair:
- `dot_scale_1`: the size scale for the first scale/cutoff pair; larger values correlate to selection of larger objects
- `dot_cut_1`: the threshold cutoff value for the first scale/cutoff pair; small cutoffs tend to yield more dots that are larger in volume; larger cutoffs tend to yield less dots that are slimmer.
- `dot_scale_2`: the size scale for the second scale/cutoff pair; this can be set to 0 if a second scale/cutoff pair isn't needed
- `dot_cut_2`: the threshold cutoff value for the second scale/cutoff pair; this can be set to 0 if a second scale/cutoff pair isn't needed
- `dot_scale_3`: the size scale for the third scale/cutoff pair; this can be set to 0 if a third scale/cutoff pair isn't needed
- `dot_cut_3`: the threshold cutoff value for the third scale/cutoff pair; this can be set to 0 if a third scale/cutoff pair isn't needed
- `dot_method`: "3D" processes the image taking into account intensities in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering any intensity in higher or lower Z planes.

In [16]:
#### USER INPUT REQUIRED ###
dot_scale_1 = 0.05
dot_cut_1 = 0.65

dot_scale_2 = 0
dot_cut_2 = 0

dot_scale_3 = 0
dot_cut_3 = 0

dot_method = '3D'

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the dot filter using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: it is helpful to adjust the scale/cutoff filters one at a time and then combine them once each pair's settings are confirmed.*

In [17]:
# segment small round structures with this (golgi vesicles)
bw_extra = dot_filter_3(struct_img,
                        dot_scale_1,
                        dot_cut_1,
                        dot_scale_2,
                        dot_cut_2,
                        dot_scale_3,
                        dot_cut_3,
                        dot_method)

# adding image to Napari as a new layer
viewer.add_image(bw_extra, scale=scale, name=f"5 - Dot filter (scale/cutoff 1 = {dot_scale_1}/{dot_cut_1}, scale/cutoff 2 = {dot_scale_2}/{dot_cut_2}, scale/cutoff 3 = {dot_scale_3}/{dot_cut_3})", opacity=0.3, colormap="green", blending='additive')

<Image layer '5 - Dot filter (scale/cutoff 1 = 0.05/0.65, scale/cutoff 2 = 0/0, scale/cutoff 3 = 0/0)' at 0x19df535f280>

### **`STEP 6` - Combine Segmentations**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block combines the results of the thinning and the dot filter together using the "logical or" operation. The resulting semantic segmentation can be viewed in Napari.

In [18]:
# combine the two segmentations together
bw_combine = np.logical_or(bw_extra, bw_thin)

# adding image to Napari as a new layer
viewer.add_image(bw_combine, scale=scale, name="6 - Combined Semantic Segmentation", opacity=0.3, blending='additive')

<Image layer '6 - Combined Semantic Segmentation' at 0x19e03c54430>

-----
## **POST-PROCESSING**

### **`STEP 7` - Remove small holes and objects**

&#x1F453; **FYI:** This code block cleans up the semantic segmentation by filling small holes and/or removing small objects that can be considered errors in the initial segmentation. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `hole_min_width`: the width of the smallest hole to be filled
- `hole_max_width`: the width of the largest hole to be filled
- `small_object_width`: the width of the largest object to be removed; any object smaller than this size will be removed
- `fill_filter_method`: "3D" processes the image taking into account segmentation values in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering the segmentation results in higher or lower Z planes.

In [21]:
#### USER INPUT REQUIRED ###
hole_min_width = 0
hole_max_width = 0

small_object_width = 2

fill_filter_method = "3D"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block fills small holes and removes small objects using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: white pixels/voxels are the ones remaining after this step*

In [22]:
# fill holes and removed small objects
cleaned_img2 = fill_and_filter_linear_size(bw_combine, 
                                           hole_min=hole_min_width, 
                                           hole_max=hole_max_width, 
                                           min_size=small_object_width,
                                           method=fill_filter_method)

# adding image to Napari as a new layer
viewer.add_image(cleaned_img2, scale=scale, name="7 - Fill holes and remove small objects", colormap="magenta", blending="additive")

<Image layer '7 - Fill holes and remove small objects [1]' at 0x19de97b2890>

-----
## **POST-POST-PROCESSING**

### **`STEP 8` - Label objects**

&#x1F453; **FYI:** This code block takes the semantic segmentation and creates an `instance segmentation`. In this output, each individual object in the image is given a unique ID number. The background pixels/voxels are still labeled as 0, but now each pixel/voxel within an object is labeled as a positive integer matching the ID number for that object. 

In this workflow objects may be separated based on `connectivity`: if a pixel/voxel is touching another pixel/voxel in any direction, they are considered the same object and each pixel/voxel within that object is labeled as the same unique ID number. Alternatively, workflow objects may be separated based on a `declumping` method—detailed in the [method_declumping](method_declumping.ipynb)—thus allowing tightly packed objects to be labeled as separate objects. Caveats to this method are also detailed in the [method_declumping](method_declumping.ipynb). 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `declump`: A True/False statement of whether to use the declumping method or not.
- `sigma`: A float value to apply in the gaussian filtering of the highpass filter. (Ignored if declump is False)
- `iterations`: The number of iterations to apply to the highpass filter. (Ignored if declump is False)
- `open_img`: A True/False statement of whether to open the highpass filtered image or not. (Ignored if declump is False)
- `thresh_adj`: A scalar to the automatically determined threshold for the highpass filter. (Ignored if declump is False)
- `min_size`: The minimum voxel count per declumped seed prior to watershedding. (Ignored if declump is False)

*Note: the [method_declumping](method_declumping.ipynb) can be used to test each of the intermediate steps included in the function below. This may be helpful when determining the appropriate settings to apply here.*

In [26]:
declump = True
dec_sig = 1.0
dec_iter = 1
dec_open = False
dec_adj = 1.0
dec_min_size = 0

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block creates an instance segmentation using the settings provided above.

*In the Napari viewer, the image is added as a "labels" layer where each object appears as a different color.*

In [28]:
# create instance segmentation based on connectivity
golgi_labels = watershed_declumping(raw_img = raw_golgi,
                                       seg_img = cleaned_img2,
                                       declump = declump,
                                       sigma = dec_sig,
                                       iterations = dec_iter,
                                       open = dec_open,
                                       thresh_adj = dec_adj,
                                       min_size = dec_min_size)

# adding image to Napari as a new layer
viewer.add_labels(golgi_labels, scale=scale, name=f"8 - Instance segmentation (declump={declump}, sigma={dec_sig}, iterations={dec_iter}, open={dec_open}, thresh_adj={dec_adj}, min_size={dec_min_size})")

<Labels layer '8 - Instance segmentation (declump=True, sigma=1.0, iterations=1, open=False, thresh_adj=1.0, min_size=0)' at 0x19df52408b0>

In [29]:
### AN ALTERNATIVE IF DECLUMPING IS NOT WORTH IT OR NEEDED ###

# # label each object with a unique integer
# golgi_labels = label(cleaned_img2)

# # adding image to Napari as a new layer
# viewer.add_labels(golgi_labels, scale=scale, name="8 - Instance segmentation")

-----
## **SAVING**

### **`Saving` - Save the segmentation output**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block saves the instance segmentation output to the `out_data_path` specified earlier.

In [46]:
# Saving file
out_file_n = export_inferred_organelle(golgi_labels, "golgi", {"file_name": file_path}, seg_data_path)
print(f"saved to: {seg_data_path}")

saved file: 03022026GLN108STOPB1T1P1-08-golgi
saved to: c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-segmentation


### **`Exporting settings` - Print out of the settings selected above for batch processing**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block reformats and prints your selected settings from above. The resulting output (found below the cell) can be copied and pasted into the `batch_process_segmentation` function.

In [38]:
[GOLGI_CH,
median_sz,
gauss_sig,
thresh_method,
thresh_adj,
cell_wise_min_area,
thin_dist_preserve,
thin_dist,
dot_scale_1,
dot_cut_1,
dot_scale_2,
dot_cut_2,
dot_scale_3,
dot_cut_3,
dot_method,
hole_min_width,
hole_max_width,
small_object_width,
fill_filter_method,
declump,
dec_sig,
dec_iter,
dec_open,
dec_adj,
dec_min_size]

[0,
 4,
 1.4,
 'median',
 1.3,
 1200,
 5,
 2,
 0.05,
 0.65,
 0,
 0,
 0,
 0,
 '3D',
 0,
 0,
 2,
 '3D',
 True,
 1.0,
 1,
 False,
 1.0,
 0]

-----
-----
## **Utilizing the `infer_golgi` function**

If you would like to process all of the steps listed above on your selected image, you can use this function. This will be the same function applied to each image during batch processing.

In [33]:
golgi_object =  infer_golgi(img_data,
                            GOLGI_CH,
                            median_sz,
                            gauss_sig,
                            thresh_method,
                            thresh_adj,
                            cell_wise_min_area,
                            thin_dist_preserve,
                            thin_dist,
                            dot_scale_1,
                            dot_cut_1,
                            dot_scale_2,
                            dot_cut_2,
                            dot_scale_3,
                            dot_cut_3,
                            dot_method,
                            hole_min_width,
                            hole_max_width,
                            small_object_width,
                            fill_filter_method,
                                declump,
                                dec_sig,
                                dec_iter,
                                dec_open,
                                dec_adj,
                                dec_min_size) 

# adding image to Napari as a new layer
viewer.add_labels(golgi_labels, scale=scale, name=f"Final Golgi segmentation")

The segmentation output here matches the output created above: False


-------------
### ✅ **INFER GOLGI COMPLETE!**

Continue to the [`batch_process_segmentation`](batch_process_segmentations.ipynb) notebook where you can apply the above settings to many images.